# Track A (time domain) pipeline, RadarML Lite hackathon

```text
Track A (time domain) pipeline for the RadarML Lite hackathon.

Method: each pulse is normalised by its own RMS and the lag-1 conjugate product
z[n] * conj(z[n-1]) (adjacent-sample phase difference weighted by magnitude) is added as two
channels. A residual 1D CNN with squeeze-excitation, a bidirectional GRU over the downsampled
sequence and attentive statistics pooling produce the class logits. No FFT, STFT, wavelet or
spectrogram representation is computed anywhere in this file.

Data: training uses the published train split only and adds noise on the fly with the
organizers' degrade() rule at the requested SNRs. Validation examples come from the bundled
make_dataset_hackathon.degrade with the `compact_dataset.py expand` seeds, so they equal the
official expanded arrays without writing the 57 GB expansion.

Project layout (paths are resolved from this file, so the script can be run from any folder):

    Radar_ML_lite/
        dataset/             recordings + split_manifest.csv / split_manifest.json   (--data)
        hackathon/           make_dataset_hackathon.py, compact_dataset.py (fallback location)
        src/track_a/track_a.py                                                       (this file)
        track_a_runs/        outputs, created on demand                              (--runs)

Run from the project root (Radar_ML_lite):

    Day 0  python src/track_a/track_a.py check
    Day 1  python src/track_a/track_a.py train --preset day1
    Day 2  python src/track_a/track_a.py train --preset day2
           python src/track_a/track_a.py select
    Day 3  python src/track_a/track_a.py benchmark --run track_a_runs/<best day2 run>
           python src/track_a/track_a.py train --config track_a_runs/selected_config.json
    Day 4  python src/track_a/track_a.py benchmark --run track_a_runs/final
           python src/track_a/track_a.py freeze --run track_a_runs/final
    Day 5  python track_a.py predict --package . --blind-dir <extracted blind_test folder>
           (inside the freeze folder)

Requires Python 3.9+, numpy, torch 2.3+ and matplotlib.
```

**Notebook notes**

- Run the code cells top to bottom once, then use the *Run* cells at the bottom (each calls `main([...])`, the same as the command line).
- Paths are found by walking up from the notebook's working directory to the folder that contains `dataset/` and `src/`.
- `--workers` defaults to 0 here: DataLoader worker processes cannot unpickle classes defined in a notebook on Windows.
- `freeze` and `predict` hash and copy `src/track_a/track_a.py`, so keep the script version beside this notebook for the Day 4/5 frozen package.

In [ ]:
from __future__ import annotations

import argparse
import copy
import csv
import dataclasses
import hashlib
import importlib.metadata
import importlib.util
import io
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

## Project paths (found from the working directory: <root>/src/track_a/)

In [ ]:
def _find_project_root() -> Path:
    """Walk up from the working directory to the folder holding dataset/ and src/ (Radar_ML_lite/)."""
    start = Path.cwd().resolve()
    for folder in (start, *start.parents):
        if (folder / "dataset").is_dir() and (folder / "src").is_dir():
            return folder
    return start


PROJECT_ROOT = _find_project_root()
SCRIPT_DIR = PROJECT_ROOT / "src" / "track_a"
CODE_FILE = SCRIPT_DIR / "track_a.py"             # script copy used for hashing/freezing (see notes)
DEFAULT_DATA_DIR = PROJECT_ROOT / "dataset"
DEFAULT_RUNS_DIR = PROJECT_ROOT / "track_a_runs"
HACKATHON_DIR = PROJECT_ROOT / "hackathon"        # bundled organizer scripts may live here


def find_bundled_script(data_dir: Path, name: str) -> Path:
    """Locate an organizer script: beside the recordings first, then in hackathon/."""
    for folder in (Path(data_dir), HACKATHON_DIR):
        if (folder / name).is_file():
            return folder / name
    raise FileNotFoundError(f"{name} not found in {data_dir} or {HACKATHON_DIR}")

## Dataset constants (README, make_dataset_hackathon.py, compact_dataset.py)

In [ ]:
CLASSES = ("barker", "cw", "fmcw", "fsk", "hyp", "nlfm", "quad")
REQUESTED_SNRS = (30, 25, 20, 15, 10, 5, 0, -5, -10, -15, -20, -25, -30)
TIE_BREAK_SNRS = (-20, -25, -30)
SAMPLE_RATE_HZ = 3.84e9 / 32
SAMPLES_PER_PULSE = 16_800
VALUES_PER_PULSE = 2 * SAMPLES_PER_PULSE
PULSES_PER_RECORDING = 100
RECORDING_SHAPE = (1, PULSES_PER_RECORDING, VALUES_PER_PULSE)
TRAIN_PER_RECORDING = 80
VALIDATION_PER_RECORDING = 20
NOISE_WINDOW = slice(0, int(20e-6 * SAMPLE_RATE_HZ))                              # 0-20 us
SIGNAL_WINDOW = slice(int(21e-6 * SAMPLE_RATE_HZ), int(119e-6 * SAMPLE_RATE_HZ))  # 21-119 us
INT16_MIN, INT16_MAX = -32768, 32767
EXPAND_SEED = 42
OFFICIAL_RELEASE = {"files": 660, "train": 52_800, "validation": 13_200}
BLIND_TEST_EXAMPLES = 171_600
FROZEN_FILES = ("track_a.py", "config.json", "model.pt")
SUBSET_RULE = "per recording, evenly spaced pulses (rounded linspace) over its published pulses in index order"
INFERENCE_COMMAND = ("python track_a.py predict --package . "
                     "--blind-dir <extracted blind_test folder> --output predictions.csv")
INFERENCE_INSTRUCTIONS = f"""\
Run inside this folder with the versions in requirements-frozen.txt:

    {INFERENCE_COMMAND}

predict checks track_a.py, config.json and model.pt against SHA256SUMS, reads test/ with the
blind package's participant_loader.py, runs the model in float32 inference mode and writes
predictions.csv (header sample_id,predicted_label) in submission_template.csv order.
Add --device cpu to run without a GPU.
"""

## Configuration

In [ ]:
@dataclasses.dataclass(frozen=True)
class Config:
    """All settings of one run. Saved as config.json and inside every checkpoint."""

    name: str
    stage: str
    seed: int = 42
    # Training data: train split only, noise added on the fly
    train_pulses_per_recording: int = TRAIN_PER_RECORDING
    train_snrs: tuple = REQUESTED_SNRS
    low_snr_weight: float = 1.0      # sampling weight of requested SNR <= -10 dB relative to the rest
    random_phase: bool = True
    # Input representation and network
    features: str = "iq_lag"         # "iq" or "iq_lag"
    stem_kernel: int = 31
    widths: tuple = (32, 64, 96, 128, 192, 256)
    blocks: tuple = (1, 2, 2, 2, 2, 2)
    kernels: tuple = (7, 7, 5, 5, 3, 3)
    temporal: str = "gru"            # "gru" or "none"
    dropout: float = 0.2
    # Optimisation
    epochs: int = 60
    batch_size: int = 128
    learning_rate: float = 1e-3
    weight_decay: float = 0.05
    warmup_epochs: float = 1.0
    min_lr_ratio: float = 0.01
    label_smoothing: float = 0.05
    logit_adjustment: float = 1.0
    grad_clip: float = 1.0
    ema_decay: float = 0.999
    # Validation subset scored during training to choose the checkpoint
    monitor_pulses_per_recording: int = 2
    monitor_every: int = 2

    def __post_init__(self) -> None:
        checks = {
            "features must be 'iq' or 'iq_lag'": self.features in ("iq", "iq_lag"),
            "temporal must be 'gru' or 'none'": self.temporal in ("gru", "none"),
            "widths, blocks and kernels need equal non-zero lengths":
                len(self.widths) == len(self.blocks) == len(self.kernels) > 0,
            "kernel sizes must be odd": all(k % 2 == 1 for k in (self.stem_kernel, *self.kernels)),
            "widths must be even": all(width % 2 == 0 for width in self.widths),
            "every stage needs at least one block": all(count >= 1 for count in self.blocks),
            "train_snrs must be requested SNRs":
                len(self.train_snrs) > 0 and set(self.train_snrs) <= set(REQUESTED_SNRS),
            "train_pulses_per_recording must be 1-80":
                1 <= self.train_pulses_per_recording <= TRAIN_PER_RECORDING,
            "monitor_pulses_per_recording must be 1-20":
                1 <= self.monitor_pulses_per_recording <= VALIDATION_PER_RECORDING,
            "epochs, batch_size and monitor_every must be positive":
                min(self.epochs, self.batch_size, self.monitor_every) >= 1,
        }
        failed = [message for message, passed in checks.items() if not passed]
        if failed:
            raise ValueError("Invalid config: " + "; ".join(failed))

    @classmethod
    def from_dict(cls, values: dict) -> Config:
        unknown = sorted(set(values) - {field.name for field in dataclasses.fields(cls)})
        if unknown:
            raise ValueError(f"Unknown config keys: {unknown}")
        return cls(**{key: tuple(value) if isinstance(value, list) else value for key, value in values.items()})

    def to_dict(self) -> dict:
        return json.loads(json.dumps(dataclasses.asdict(self)))


def _build_presets() -> dict[str, Config]:
    """Day 2 changes one factor at a time from the baseline, all with the same seed."""
    baseline = Config(name="day2-baseline", stage="day2", train_pulses_per_recording=40, epochs=20)
    day2 = (
        baseline,
        dataclasses.replace(baseline, name="day2-iq-only", features="iq"),
        dataclasses.replace(baseline, name="day2-short-stem", stem_kernel=7),
        dataclasses.replace(baseline, name="day2-no-gru", temporal="none"),
        dataclasses.replace(baseline, name="day2-low-snr-x2", low_snr_weight=2.0),
        dataclasses.replace(baseline, name="day2-wide", widths=(48, 96, 144, 192, 288, 384)),
    )
    presets = {config.name: config for config in day2}
    presets["day1"] = dataclasses.replace(
        baseline, name="day1", stage="day1", train_pulses_per_recording=10, epochs=8, monitor_every=4)
    presets["final"] = dataclasses.replace(
        baseline, name="final", stage="final", train_pulses_per_recording=TRAIN_PER_RECORDING, epochs=60,
        monitor_pulses_per_recording=4)
    return presets


PRESETS = _build_presets()
PRESET_GROUPS = {"day2": tuple(name for name in PRESETS if name.startswith("day2-"))}

## Utilities

In [ ]:
def read_json(path: Path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path: Path, data) -> None:
    Path(path).write_text(json.dumps(data, indent=2) + "\n", encoding="utf-8")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def format_duration(seconds: float) -> str:
    minutes = int(round(seconds / 60))
    return f"{minutes // 60}h{minutes % 60:02d}m"


def format_bytes(count: int | None) -> str:
    if count is None:
        return "n/a"
    if count >= 2**30:
        return f"{count / 2**30:.2f} GiB"
    return f"{count / 2**20:.1f} MiB"


def import_from_path(path: Path):
    """Import a script by file path (bundled dataset scripts, the blind-test loader)."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}")
    spec = importlib.util.spec_from_file_location(path.stem, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def resolve_device(name: str | None) -> torch.device:
    if name:
        return torch.device(name)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def resolve_runs_dir(args: argparse.Namespace) -> Path:
    if args.runs:
        return Path(args.runs).resolve()
    return DEFAULT_RUNS_DIR


def to_device(tensor: torch.Tensor, device: torch.device) -> torch.Tensor:
    if device.type == "cuda":
        return tensor.pin_memory().to(device, non_blocking=True)
    return tensor.to(device)


def configure_backends(inference: bool) -> None:
    """Autotuned kernels for training; deterministic full-float32 kernels for scoring and inference."""
    torch.backends.cudnn.benchmark = not inference
    torch.backends.cudnn.deterministic = inference
    torch.backends.cudnn.allow_tf32 = not inference
    torch.backends.cuda.matmul.allow_tf32 = False


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

## Development data

In [ ]:
@dataclasses.dataclass(frozen=True)
class PulseRef:
    """One source pulse. Filenames and metadata manage examples and labels, never model inputs."""

    file: str          # compact recording, as listed in split_manifest.csv
    pulse_index: int
    label: int         # index into CLASSES
    source_file: str   # original filename from the JSON sidecar; seeds the official noise

    @property
    def duration(self) -> str:
        return self.file.split("_")[0]


def label_from_filename(filename: str) -> str:
    """Waveform token of the filename: Barker subtypes -> barker, fsk#0..fsk#3 -> fsk."""
    token = filename.split("_")[1].split("#")[0].lower()
    label = "barker" if token.startswith("barker") else token
    if label not in CLASSES:
        raise ValueError(f"Unrecognised modulation token in {filename}")
    return label


def verify_manifest_checksum(data_dir: Path) -> str:
    """Compare split_manifest.csv with the SHA256 recorded in split_manifest.json.

    The published CSV has CRLF line endings. A download or Git checkout that converts them to LF
    changes the hash but not the content; that case is accepted and reported.
    """
    expected = read_json(data_dir / "split_manifest.json")["manifest_sha256"]
    content = (data_dir / "split_manifest.csv").read_bytes()
    if hashlib.sha256(content).hexdigest() == expected:
        return "exact match"
    if hashlib.sha256(content.replace(b"\r\n", b"\n").replace(b"\n", b"\r\n")).hexdigest() == expected:
        return "match after restoring CRLF line endings (content identical)"
    raise ValueError("split_manifest.csv does not match split_manifest.json; download the raw file again")


def read_manifest(data_dir: Path) -> list[dict]:
    with (data_dir / "split_manifest.csv").open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["file", "pulse_index", "split"]:
            raise ValueError(f"Unexpected split_manifest.csv columns: {reader.fieldnames}")
        return list(reader)


def load_split(data_dir: Path) -> dict[str, list[PulseRef]]:
    """Published train and validation assignments with labels and original source filenames."""
    verify_manifest_checksum(data_dir)
    split: dict[str, list[PulseRef]] = {"train": [], "validation": []}
    source_files: dict[str, str] = {}
    for row in read_manifest(data_dir):
        file = row["file"]
        if file not in source_files:
            source_files[file] = read_json((data_dir / file).with_suffix(".json"))["source_file"]
        label = CLASSES.index(label_from_filename(file))
        split[row["split"]].append(PulseRef(file, int(row["pulse_index"]), label, source_files[file]))
    return split


def dataset_identity(data_dir: Path) -> dict:
    """Identifies the development data a run used."""
    published = read_json(data_dir / "split_manifest.json")
    identity = dict(data_dir=str(data_dir), manifest_sha256=published["manifest_sha256"],
                    split_seed=published["seed"])
    if (data_dir / "verification.json").is_file():
        identity["compact_verification"] = read_json(data_dir / "verification.json")
    return identity


def evenly_spaced(refs: Sequence[PulseRef], per_recording: int) -> list[PulseRef]:
    """Subset rule: per recording, evenly spaced pulses in pulse-index order (deterministic)."""
    by_file: dict[str, list[PulseRef]] = defaultdict(list)
    for ref in refs:
        by_file[ref.file].append(ref)
    subset = []
    for file in sorted(by_file):
        group = sorted(by_file[file], key=lambda ref: ref.pulse_index)
        if per_recording > len(group):
            raise ValueError(f"{file}: {per_recording} pulses requested, {len(group)} available")
        positions = np.round(np.linspace(0, len(group) - 1, per_recording)).astype(int)
        subset.extend(group[position] for position in positions)
    return subset


def load_pulses(data_dir: Path, refs: Sequence[PulseRef]) -> np.ndarray:
    """Copy clean recorded pulses into one int16 array of shape (N, 33600)."""
    pulses = np.empty((len(refs), VALUES_PER_PULSE), dtype=np.int16)
    rows_by_file: dict[str, list[tuple[int, int]]] = defaultdict(list)
    for row, ref in enumerate(refs):
        rows_by_file[ref.file].append((row, ref.pulse_index))
    for count, (file, items) in enumerate(sorted(rows_by_file.items()), 1):
        recording = np.load(data_dir / file, mmap_mode="r", allow_pickle=False)
        if recording.shape != RECORDING_SHAPE or recording.dtype != np.int16:
            raise ValueError(f"{file}: expected int16 {RECORDING_SHAPE}, found {recording.dtype} {recording.shape}")
        rows, indices = zip(*items)
        pulses[list(rows)] = recording[0, list(indices)]
        del recording
        if count % 110 == 0 or count == len(rows_by_file):
            print(f"  read {count}/{len(rows_by_file)} recordings", flush=True)
    return pulses


def expand_noise_seed(source_file: str, snr_db: int, pulse_index: int) -> int:
    """Noise seed of make_dataset_hackathon.process_rows as called by `compact_dataset.py expand`."""
    key = f"{EXPAND_SEED}_{Path(source_file).name}_{snr_db}_{pulse_index}"
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16)


class OfficialNoiseDataset(Dataset):
    """Validation examples identical to `compact_dataset.py expand` output (same NumPy version).

    Item k is pulse k // len(snrs) at snrs[k % len(snrs)], so consecutive items share a recording.
    Items are (interleaved int16 values, item index, unattainable-target flag, clipped value count).
    """

    def __init__(self, data_dir: Path, refs: Sequence[PulseRef], snrs: Sequence[int]):
        self.data_dir = Path(data_dir)
        self.refs = list(refs)
        self.snrs = [int(snr) for snr in snrs]
        self._degrade = None
        self._file = None
        self._recording = None

    def __getstate__(self) -> dict:
        # Worker processes import the bundled script and open their own memory maps.
        return {**self.__dict__, "_degrade": None, "_file": None, "_recording": None}

    def __len__(self) -> int:
        return len(self.refs) * len(self.snrs)

    def __getitem__(self, index: int):
        ref = self.refs[index // len(self.snrs)]
        snr = self.snrs[index % len(self.snrs)]
        if self._degrade is None:
            self._degrade = import_from_path(find_bundled_script(self.data_dir, "make_dataset_hackathon.py")).degrade
        if self._file != ref.file:
            self._recording = np.load(self.data_dir / ref.file, mmap_mode="r", allow_pickle=False)
            self._file = ref.file
        rng = np.random.default_rng(expand_noise_seed(ref.source_file, snr, ref.pulse_index))
        values, unattainable, clipped = self._degrade(self._recording[0, ref.pulse_index], snr, rng)
        return torch.from_numpy(values), index, bool(unattainable), int(clipped)

## Signal processing (time domain only)

In [ ]:
def interleaved_to_iq(values: torch.Tensor) -> torch.Tensor:
    """(B, 33600) interleaved I/Q -> (B, 2, 16800) float32 [I, Q] in ADC units."""
    return values.float().reshape(values.shape[0], SAMPLES_PER_PULSE, 2).transpose(1, 2).contiguous()


def add_requested_noise(iq: torch.Tensor, snr_db: torch.Tensor, generator: torch.Generator) -> torch.Tensor:
    """Torch port of make_dataset_hackathon.degrade for a batch of (B, 2, L) float I/Q.

    Same estimation windows, unattainable-target rule (no noise added), int16 clipping and
    truncation toward zero; only the noise realisation differs.
    """
    power = iq.square().sum(dim=1)
    noise_power = power[:, NOISE_WINDOW].mean(dim=1)
    signal_power = power[:, SIGNAL_WINDOW].mean(dim=1) - noise_power
    added_power = signal_power / torch.pow(10.0, snr_db / 10.0) - noise_power
    sigma = torch.sqrt(added_power.clamp(min=0.0) / 2.0).view(-1, 1, 1)
    noise = torch.randn(iq.shape, generator=generator, device=iq.device, dtype=iq.dtype)
    return (iq + sigma * noise).clamp(INT16_MIN, INT16_MAX).trunc()


def rotate_phase(iq: torch.Tensor, generator: torch.Generator) -> torch.Tensor:
    """Multiply each pulse by exp(j*theta), theta ~ U[0, 2*pi). Carrier phase is not a class cue."""
    theta = torch.rand((iq.shape[0], 1), generator=generator, device=iq.device) * (2.0 * math.pi)
    cos, sin = torch.cos(theta), torch.sin(theta)
    i, q = iq[:, 0], iq[:, 1]
    return torch.stack((i * cos - q * sin, i * sin + q * cos), dim=1)


def pulse_features(iq: torch.Tensor, kind: str) -> torch.Tensor:
    """Per-pulse RMS normalisation, plus the lag-1 conjugate product when kind == "iq_lag".

    z[n] * conj(z[n-1]) has the adjacent-sample phase difference as its angle and |z[n]| |z[n-1]|
    as its magnitude, so noise-dominated samples carry little weight. Computed in float32.
    """
    iq = iq.float()
    mean_power = iq.square().sum(dim=1, keepdim=True).mean(dim=2, keepdim=True)
    z = iq * torch.rsqrt(mean_power.clamp(min=1e-12))
    if kind == "iq":
        return z
    i, q = z[:, 0], z[:, 1]
    real = i[:, 1:] * i[:, :-1] + q[:, 1:] * q[:, :-1]
    imag = q[:, 1:] * i[:, :-1] - i[:, 1:] * q[:, :-1]
    lag = F.pad(torch.stack((real, imag), dim=1), (1, 0))
    return torch.cat((z, lag), dim=1)

## Network

In [ ]:
class SqueezeExcite(nn.Module):
    """Channel gating from the time-averaged response (Hu et al., 2018)."""

    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        hidden = max(8, channels // reduction)
        self.gate = nn.Sequential(nn.Linear(channels, hidden), nn.GELU(), nn.Linear(hidden, channels), nn.Sigmoid())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.gate(x.mean(dim=2)).unsqueeze(2)


class ResidualBlock(nn.Module):
    """Two convolutions with squeeze-excitation; average-pool shortcut when downsampling."""

    def __init__(self, in_channels: int, out_channels: int, kernel: int, stride: int):
        super().__init__()
        padding = kernel // 2
        self.body = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel, stride, padding, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
            nn.Conv1d(out_channels, out_channels, kernel, 1, padding, bias=False),
            nn.BatchNorm1d(out_channels),
            SqueezeExcite(out_channels),
        )
        nn.init.zeros_(self.body[4].weight)  # each block starts close to its shortcut
        if stride == 1 and in_channels == out_channels:
            self.shortcut = nn.Identity()
        else:
            self.shortcut = nn.Sequential(
                nn.AvgPool1d(stride, stride, ceil_mode=True) if stride > 1 else nn.Identity(),
                nn.Conv1d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm1d(out_channels),
            )
        self.activation = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(self.body(x) + self.shortcut(x))


class TemporalGRU(nn.Module):
    """Bidirectional GRU over the downsampled sequence: models the order of local features."""

    def __init__(self, channels: int):
        super().__init__()
        self.gru = nn.GRU(channels, channels // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        with torch.autocast(device_type=x.device.type, enabled=False):
            sequence = x.float().transpose(1, 2)
            self.gru.flatten_parameters()
            output, _ = self.gru(sequence)
            return self.norm(output + sequence).transpose(1, 2)


class AttentiveStatsPool(nn.Module):
    """Attention-weighted mean and standard deviation over time (Okabe et al., 2018)."""

    def __init__(self, channels: int, hidden: int = 128):
        super().__init__()
        self.attention = nn.Sequential(nn.Conv1d(channels, hidden, 1), nn.Tanh(), nn.Conv1d(hidden, channels, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        with torch.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            weights = torch.softmax(self.attention(x), dim=2)
            mean = (weights * x).sum(dim=2)
            variance = (weights * x.square()).sum(dim=2) - mean.square()
            return torch.cat((mean, variance.clamp(min=1e-5).sqrt()), dim=1)


class TrackANet(nn.Module):
    """(B, 2, 16800) raw I/Q in ADC units -> (B, 7) logits. Preprocessing is part of the model."""

    def __init__(self, config: Config):
        super().__init__()
        self.feature_kind = config.features
        in_channels = 2 if config.features == "iq" else 4
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, config.widths[0], config.stem_kernel, 2, config.stem_kernel // 2, bias=False),
            nn.BatchNorm1d(config.widths[0]),
            nn.GELU(),
        )
        layers, channels = [], config.widths[0]
        for width, count, kernel in zip(config.widths, config.blocks, config.kernels):
            for block in range(count):
                layers.append(ResidualBlock(channels, width, kernel, stride=2 if block == 0 else 1))
                channels = width
        self.encoder = nn.Sequential(*layers)
        self.temporal = TemporalGRU(channels) if config.temporal == "gru" else nn.Identity()
        self.pool = AttentiveStatsPool(channels)
        self.head = nn.Sequential(nn.Linear(2 * channels, channels), nn.GELU(), nn.Dropout(config.dropout),
                                  nn.Linear(channels, len(CLASSES)))

    def forward(self, iq: torch.Tensor) -> torch.Tensor:
        x = self.encoder(self.stem(pulse_features(iq, self.feature_kind)))
        return self.head(self.pool(self.temporal(x)))

## Scoring and reports (same definitions as score_submission.py)

In [ ]:
def confusion_matrix(labels: np.ndarray, predictions: np.ndarray) -> np.ndarray:
    """Rows are true labels, columns predicted labels, both in CLASSES order."""
    n = len(CLASSES)
    return np.bincount(labels * n + predictions, minlength=n * n).reshape(n, n)


def balanced_accuracy(matrix: np.ndarray, require_all: bool = True) -> float:
    """Mean per-class recall."""
    support = matrix.sum(axis=1)
    if require_all and np.any(support == 0):
        raise ValueError("Every class must be present at every SNR")
    present = support > 0
    return float(np.mean(matrix.diagonal()[present] / support[present]))


def score_report(labels: np.ndarray, predictions: np.ndarray, snrs: Sequence[int]) -> dict:
    """labels (P,), predictions (P, S): balanced accuracy per requested SNR and their mean."""
    per_snr = []
    for column, snr in sorted(enumerate(snrs), key=lambda item: -item[1]):
        matrix = confusion_matrix(labels, predictions[:, column])
        per_snr.append(dict(requested_snr_db=int(snr), balanced_accuracy=balanced_accuracy(matrix),
                            accuracy=float(matrix.trace() / matrix.sum()), confusion_matrix=matrix.tolist()))
    by_snr = {row["requested_snr_db"]: row["balanced_accuracy"] for row in per_snr}
    tie_break = [by_snr[snr] for snr in TIE_BREAK_SNRS if snr in by_snr]
    return dict(classes=list(CLASSES), samples=int(predictions.size),
                mean_balanced_accuracy=float(np.mean(list(by_snr.values()))),
                low_snr_mean_balanced_accuracy=float(np.mean(tie_break)) if tie_break else None,
                per_snr=per_snr)


def hardest_confusions(per_snr: Sequence[dict], max_snr: int = -10, top: int = 8) -> list[dict]:
    """Largest off-diagonal confusion rates, pooled over requested SNRs <= max_snr."""
    pooled = sum(np.array(row["confusion_matrix"]) for row in per_snr if row["requested_snr_db"] <= max_snr)
    rates = pooled / pooled.sum(axis=1, keepdims=True)
    pairs = [(rates[t, p], t, p) for t in range(len(CLASSES)) for p in range(len(CLASSES)) if t != p]
    return [dict(true=CLASSES[t], predicted=CLASSES[p], rate=float(rate))
            for rate, t, p in sorted(pairs, reverse=True)[:top]]


def balanced_accuracy_by_duration(refs: Sequence[PulseRef], labels: np.ndarray, predictions: np.ndarray) -> dict:
    """Robustness view: balanced accuracy per requested SNR for each pulse duration."""
    durations = np.array([ref.duration for ref in refs])
    result = {}
    for duration in sorted(set(durations.tolist()), key=lambda value: float(value.rstrip("us"))):
        mask = durations == duration
        result[duration] = {
            str(snr): balanced_accuracy(confusion_matrix(labels[mask], predictions[mask, column]), require_all=False)
            for column, snr in enumerate(REQUESTED_SNRS)}
    return result


def build_report(refs: Sequence[PulseRef], predictions: np.ndarray, diagnostics: list[dict]) -> dict:
    labels = np.array([ref.label for ref in refs], dtype=np.int64)
    report = score_report(labels, predictions, REQUESTED_SNRS)
    report["hardest_confusions_at_or_below_minus_10_db"] = hardest_confusions(report["per_snr"])
    report["balanced_accuracy_by_duration"] = balanced_accuracy_by_duration(refs, labels, predictions)
    report["preprocessing_diagnostics"] = diagnostics
    report["validation_noise"] = "make_dataset_hackathon.degrade with compact_dataset.py expand seeds (seed 42)"
    return report


def print_report(report: dict) -> None:
    print(f"Validation mean balanced accuracy {report['mean_balanced_accuracy']:.4f}; "
          f"mean at -20/-25/-30 dB {report['low_snr_mean_balanced_accuracy']:.4f}")
    for row in report["per_snr"]:
        print(f"  {row['requested_snr_db']:+4d} dB  balanced accuracy {row['balanced_accuracy']:.4f}"
              f"  accuracy {row['accuracy']:.4f}")
    confusions = report["hardest_confusions_at_or_below_minus_10_db"][:5]
    print("Most frequent confusions at <= -10 dB: "
          + ", ".join(f"{c['true']}->{c['predicted']} {c['rate']:.1%}" for c in confusions), flush=True)


def write_validation_predictions(path: Path, refs: Sequence[PulseRef], predictions: np.ndarray) -> None:
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(["file", "pulse_index", "requested_snr_db", "label", "predicted_label"])
        for ref, row in zip(refs, predictions):
            for snr, predicted in zip(REQUESTED_SNRS, row):
                writer.writerow([ref.file, ref.pulse_index, snr, CLASSES[ref.label], CLASSES[predicted]])


def pyplot():
    """Matplotlib with a file-only backend, imported lazily so inference does not need it."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    return plt


def save_report_plots(report: dict, run_dir: Path, suffix: str = "") -> None:
    """Balanced accuracy and per-class recall against requested SNR; one confusion matrix per SNR."""
    plt = pyplot()
    rows = sorted(report["per_snr"], key=lambda row: row["requested_snr_db"])
    snrs = [row["requested_snr_db"] for row in rows]
    matrices = [np.array(row["confusion_matrix"], dtype=float) for row in rows]
    recalls = np.array([matrix.diagonal() / matrix.sum(axis=1) for matrix in matrices])

    fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
    for column, name in enumerate(CLASSES):
        ax.plot(snrs, recalls[:, column], linewidth=1, alpha=0.75, label=f"{name} recall")
    ax.plot(snrs, [row["balanced_accuracy"] for row in rows], color="black", linewidth=2.5, marker="o",
            label="balanced accuracy")
    ax.axhline(1 / len(CLASSES), color="grey", linestyle="--", linewidth=1, label="chance (1/7)")
    ax.set(xlabel="Requested SNR (dB)", ylabel="Recall / balanced accuracy", xticks=snrs, ylim=(0, 1.02),
           title=f"Track A validation: mean balanced accuracy {report['mean_balanced_accuracy']:.4f}")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2, loc="lower right")
    fig.savefig(run_dir / f"ba_vs_snr{suffix}.png", dpi=150)
    plt.close(fig)

    fig, axes = plt.subplots(4, 4, figsize=(15, 15), layout="constrained")
    for ax, row, matrix in zip(axes.flat, reversed(rows), reversed(matrices)):
        rates = matrix / matrix.sum(axis=1, keepdims=True)
        ax.imshow(rates, vmin=0, vmax=1, cmap="Blues")
        for (true, predicted), rate in np.ndenumerate(rates):
            ax.text(predicted, true, f"{100 * rate:.0f}", ha="center", va="center", fontsize=6,
                    color="white" if rate > 0.5 else "black")
        ax.set_title(f"{row['requested_snr_db']:+d} dB, BA {row['balanced_accuracy']:.3f}", fontsize=9)
        ax.set_xticks(range(len(CLASSES)))
        ax.set_xticklabels(CLASSES, rotation=90, fontsize=7)
        ax.set_yticks(range(len(CLASSES)))
        ax.set_yticklabels(CLASSES, fontsize=7)
    for ax in axes.flat[len(rows):]:
        ax.axis("off")
    fig.supxlabel("Predicted label (row-normalised %)")
    fig.supylabel("True label")
    fig.savefig(run_dir / f"confusion_matrices{suffix}.png", dpi=130)
    plt.close(fig)


def save_training_curves(run_dir: Path) -> None:
    with (run_dir / "training_log.csv").open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    monitored = [row for row in rows if row["monitor_mean_ba"]]
    epochs = [int(row["epoch"]) for row in monitored]
    plt = pyplot()
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
    left.plot([int(row["epoch"]) for row in rows], [float(row["train_loss"]) for row in rows])
    left.set(xlabel="Epoch", ylabel="Logit-adjusted cross-entropy", title="Training loss")
    right.plot(epochs, [float(row["monitor_mean_ba"]) for row in monitored], marker="o", label="mean over 13 SNRs")
    right.plot(epochs, [float(row["monitor_low_snr_ba"]) for row in monitored], marker="o",
               label="mean at -20/-25/-30 dB")
    right.set(xlabel="Epoch", ylabel="Balanced accuracy", title="Monitored validation subset")
    right.legend()
    for ax in (left, right):
        ax.grid(alpha=0.3)
    fig.savefig(run_dir / "training_curves.png", dpi=130)
    plt.close(fig)

## Training

In [ ]:
class ModelEma:
    """Exponential moving average of parameters and buffers, with warm-up of the decay."""

    def __init__(self, model: nn.Module, decay: float):
        self.module = copy.deepcopy(model).eval().requires_grad_(False)
        self.decay = decay
        self.updates = 0

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        self.updates += 1
        decay = min(self.decay, (1 + self.updates) / (10 + self.updates))
        for average, current in zip(self.module.state_dict().values(), model.state_dict().values()):
            if average.dtype.is_floating_point:
                average.lerp_(current, 1.0 - decay)
            else:
                average.copy_(current)


def cosine_with_warmup(config: Config, steps_per_epoch: int):
    """Learning-rate factor per step: linear warm-up, then cosine decay to min_lr_ratio."""
    total = config.epochs * steps_per_epoch
    warmup = max(1, round(config.warmup_epochs * steps_per_epoch))

    def factor(step: int) -> float:
        if step < warmup:
            return (step + 1) / warmup
        progress = min(1.0, (step - warmup) / max(1, total - warmup))
        return config.min_lr_ratio + (1.0 - config.min_lr_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))

    return factor


def parameter_groups(model: nn.Module, weight_decay: float) -> list[dict]:
    """Decay weight matrices and kernels; leave biases and normalisation scales undecayed."""
    decay = [parameter for parameter in model.parameters() if parameter.ndim > 1]
    no_decay = [parameter for parameter in model.parameters() if parameter.ndim <= 1]
    return [dict(params=decay, weight_decay=weight_decay), dict(params=no_decay, weight_decay=0.0)]


def snr_probabilities(config: Config) -> list[float]:
    weights = [config.low_snr_weight if snr <= -10 else 1.0 for snr in config.train_snrs]
    return [weight / sum(weights) for weight in weights]


def training_precision(device: torch.device) -> torch.dtype | None:
    """bfloat16 autocast where supported, float16 with loss scaling otherwise, float32 on CPU."""
    if device.type != "cuda":
        return None
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


class Trainer:
    """Optimisation state of one run. SNR, phase and noise draws use one seeded device generator."""

    def __init__(self, config: Config, device: torch.device, class_counts: np.ndarray, steps_per_epoch: int):
        self.config, self.device, self.steps_per_epoch = config, device, steps_per_epoch
        self.model = TrackANet(config).to(device)
        self.ema = ModelEma(self.model, config.ema_decay)
        self.optimizer = torch.optim.AdamW(parameter_groups(self.model, config.weight_decay), lr=config.learning_rate)
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(self.optimizer, cosine_with_warmup(config, steps_per_epoch))
        self.amp_dtype = training_precision(device)
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.amp_dtype == torch.float16)
        self.generator = torch.Generator(device=device)
        self.generator.manual_seed(config.seed)
        self.order_rng = np.random.default_rng(config.seed)
        # Logit adjustment (Menon et al., 2021): training priors only, raw logits at inference.
        prior = torch.as_tensor(class_counts / class_counts.sum(), dtype=torch.float32, device=device)
        self.log_prior = config.logit_adjustment * prior.log()
        self.snr_values = torch.tensor(config.train_snrs, dtype=torch.float32, device=device)
        self.snr_weights = torch.tensor(snr_probabilities(config), dtype=torch.float32, device=device)

    def noisy_inputs(self, pulses: torch.Tensor) -> torch.Tensor:
        """Clean int16 training pulses -> noisy float I/Q at randomly drawn requested SNRs."""
        draws = torch.multinomial(self.snr_weights, len(pulses), replacement=True, generator=self.generator)
        iq = interleaved_to_iq(pulses)
        if self.config.random_phase:
            iq = rotate_phase(iq, self.generator)
        return add_requested_noise(iq, self.snr_values[draws], self.generator)

    def train_epoch(self, train_x: np.ndarray, train_y: np.ndarray) -> tuple[float, float]:
        config, device = self.config, self.device
        self.model.train()
        order = self.order_rng.permutation(len(train_x))
        loss_sum = torch.zeros((), device=device)
        correct = torch.zeros((), dtype=torch.long, device=device)
        for step in range(self.steps_per_epoch):
            index = np.sort(order[step * config.batch_size:(step + 1) * config.batch_size])
            inputs = self.noisy_inputs(to_device(torch.from_numpy(train_x[index]), device))
            targets = to_device(torch.from_numpy(train_y[index]), device)
            with torch.autocast(device_type=device.type, dtype=self.amp_dtype or torch.bfloat16,
                                enabled=self.amp_dtype is not None):
                logits = self.model(inputs)
            loss = F.cross_entropy(logits.float() + self.log_prior, targets, label_smoothing=config.label_smoothing)
            self.optimizer.zero_grad(set_to_none=True)
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            nn.utils.clip_grad_norm_(self.model.parameters(), config.grad_clip)
            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.scheduler.step()
            self.ema.update(self.model)
            loss_sum += loss.detach() * len(index)
            correct += (logits.argmax(dim=1) == targets).sum()
        seen = self.steps_per_epoch * config.batch_size
        return loss_sum.item() / seen, correct.item() / seen


@torch.no_grad()
def monitor_score(model: nn.Module, pulses: np.ndarray, labels: np.ndarray, device: torch.device,
                  seed: int, batch_size: int = 256) -> dict:
    """Validation subset at all 13 requested SNRs with fixed-seed noise from the same rule."""
    model.eval()
    generator = torch.Generator(device=device)
    generator.manual_seed(seed)
    predictions = np.empty((len(pulses), len(REQUESTED_SNRS)), dtype=np.int64)
    for column, snr in enumerate(REQUESTED_SNRS):
        for start in range(0, len(pulses), batch_size):
            batch = to_device(torch.from_numpy(pulses[start:start + batch_size]), device)
            snr_db = torch.full((len(batch),), float(snr), device=device)
            iq = add_requested_noise(interleaved_to_iq(batch), snr_db, generator)
            predictions[start:start + len(batch), column] = model(iq).argmax(dim=1).cpu().numpy()
    return score_report(labels, predictions, REQUESTED_SNRS)


def save_checkpoint(path: Path, model: nn.Module, config: Config, epoch: int, score: float | None) -> None:
    torch.save(dict(config=config.to_dict(), state_dict=model.state_dict(), epoch=epoch, monitor_score=score), path)


def load_checkpoint(path: Path, device: torch.device) -> tuple[Config, TrackANet, dict]:
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    config = Config.from_dict(checkpoint["config"])
    model = TrackANet(config)
    model.load_state_dict(checkpoint["state_dict"])
    return config, model.to(device).eval(), checkpoint


def validation_summary(report: dict) -> dict:
    return dict(mean_balanced_accuracy=report["mean_balanced_accuracy"],
                low_snr_mean_balanced_accuracy=report["low_snr_mean_balanced_accuracy"],
                per_snr={str(row["requested_snr_db"]): row["balanced_accuracy"] for row in report["per_snr"]})


EXPERIMENT_FIELDS = (
    "run", "stage", "finished_utc", "mean_balanced_accuracy", "low_snr_mean_balanced_accuracy",
    *(f"ba_{snr}" for snr in REQUESTED_SNRS), "features", "stem_kernel", "temporal", "low_snr_weight",
    "widths", "train_pulses_per_recording", "epochs", "seed", "parameters", "training_seconds")


def append_experiment(runs_dir: Path, summary: dict) -> None:
    """Experiment log: one line per finished run with subset, seed, settings, score and runtime."""
    config, validation = summary["config"], summary["validation"]
    row = dict(
        run=summary["run"], stage=summary["stage"], finished_utc=summary["finished_utc"],
        mean_balanced_accuracy=f"{validation['mean_balanced_accuracy']:.5f}",
        low_snr_mean_balanced_accuracy=f"{validation['low_snr_mean_balanced_accuracy']:.5f}",
        features=config["features"], stem_kernel=config["stem_kernel"], temporal=config["temporal"],
        low_snr_weight=config["low_snr_weight"], widths=" ".join(map(str, config["widths"])),
        train_pulses_per_recording=config["train_pulses_per_recording"], epochs=config["epochs"],
        seed=config["seed"], parameters=summary["parameters"], training_seconds=summary["training_seconds"])
    row.update({f"ba_{snr}": f"{value:.4f}" for snr, value in validation["per_snr"].items()})
    path = runs_dir / "experiments.csv"
    is_new = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=EXPERIMENT_FIELDS)
        if is_new:
            writer.writeheader()
        writer.writerow(row)


def record_validation(run_dir: Path, report: dict) -> None:
    """Store best-checkpoint validation scores in summary.json; log the run once in experiments.csv."""
    path = run_dir / "summary.json"
    if not path.is_file():
        return
    summary = read_json(path)
    first_record = "validation" not in summary
    summary["validation"] = validation_summary(report)
    write_json(path, summary)
    if first_record:
        append_experiment(run_dir.parent, summary)


def train_run(config: Config, data_dir: Path, runs_dir: Path, device: torch.device, workers: int) -> None:
    """Train one configuration, then score its best checkpoint on the full validation split."""
    run_dir = runs_dir / config.name
    if run_dir.exists():
        raise SystemExit(f"{run_dir} exists without validation results. If training finished, run "
                         f"`evaluate --run {run_dir}`; otherwise remove the folder or pass --name.")
    run_dir.mkdir(parents=True)
    write_json(run_dir / "config.json", config.to_dict())
    write_json(run_dir / "environment.json", environment_info(device))
    print(f"\n=== {config.name} ({config.stage}) on {device} ===", flush=True)
    set_seed(config.seed)
    configure_backends(inference=False)

    started = time.perf_counter()
    split = load_split(data_dir)
    train_refs = evenly_spaced(split["train"], config.train_pulses_per_recording)
    monitor_refs = evenly_spaced(split["validation"], config.monitor_pulses_per_recording)
    train_y = np.array([ref.label for ref in train_refs], dtype=np.int64)
    monitor_y = np.array([ref.label for ref in monitor_refs], dtype=np.int64)
    class_counts = np.bincount(train_y, minlength=len(CLASSES))
    if np.any(class_counts == 0):
        raise ValueError("The training subset must contain every class")
    print("Training pulses per class: " + ", ".join(f"{c} {n}" for c, n in zip(CLASSES, class_counts)), flush=True)
    print(f"Loading {len(train_refs)} training and {len(monitor_refs)} monitor pulses", flush=True)
    train_x = load_pulses(data_dir, train_refs)
    monitor_x = load_pulses(data_dir, monitor_refs)
    load_seconds = time.perf_counter() - started

    steps_per_epoch = len(train_x) // config.batch_size
    if steps_per_epoch == 0:
        raise ValueError("batch_size is larger than the training subset")
    trainer = Trainer(config, device, class_counts, steps_per_epoch)
    parameters = sum(parameter.numel() for parameter in trainer.model.parameters())
    precision = str(trainer.amp_dtype or torch.float32).replace("torch.", "")
    print(f"Model {parameters:,} parameters; {precision}; {steps_per_epoch} steps per epoch", flush=True)

    log_path = run_dir / "training_log.csv"
    log_fields = ["epoch", "learning_rate", "train_loss", "train_accuracy", "epoch_seconds",
                  "monitor_mean_ba", "monitor_low_snr_ba", *(f"ba_{snr}" for snr in REQUESTED_SNRS)]
    with log_path.open("w", newline="", encoding="utf-8") as stream:
        csv.DictWriter(stream, fieldnames=log_fields).writeheader()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    best_score, best_epoch = -1.0, 0
    train_started = time.perf_counter()
    for epoch in range(1, config.epochs + 1):
        epoch_started = time.perf_counter()
        loss, accuracy = trainer.train_epoch(train_x, train_y)
        row = dict.fromkeys(log_fields, "")
        row.update(epoch=epoch, learning_rate=f"{trainer.scheduler.get_last_lr()[0]:.3e}",
                   train_loss=f"{loss:.5f}", train_accuracy=f"{accuracy:.4f}")
        message = f"epoch {epoch:3d}/{config.epochs}  loss {loss:.4f}  train acc {accuracy:.3f}"
        if epoch % config.monitor_every == 0 or epoch == config.epochs:
            monitored = monitor_score(trainer.ema.module, monitor_x, monitor_y, device, seed=config.seed + 1)
            score, low = monitored["mean_balanced_accuracy"], monitored["low_snr_mean_balanced_accuracy"]
            row.update(monitor_mean_ba=f"{score:.5f}", monitor_low_snr_ba=f"{low:.5f}")
            row.update({f"ba_{r['requested_snr_db']}": f"{r['balanced_accuracy']:.4f}" for r in monitored["per_snr"]})
            message += f"  monitor BA {score:.4f} (-20/-25/-30 dB {low:.4f})"
            if score > best_score:
                best_score, best_epoch = score, epoch
                save_checkpoint(run_dir / "checkpoint_best.pt", trainer.ema.module, config, epoch, score)
                message += " best"
        seconds = time.perf_counter() - epoch_started
        row["epoch_seconds"] = f"{seconds:.1f}"
        with log_path.open("a", newline="", encoding="utf-8") as stream:
            csv.DictWriter(stream, fieldnames=log_fields).writerow(row)
        remaining = (time.perf_counter() - train_started) / epoch * (config.epochs - epoch)
        print(f"{message}  {seconds:.0f} s  ETA {format_duration(remaining)}", flush=True)
    training_seconds = time.perf_counter() - train_started
    save_checkpoint(run_dir / "checkpoint_last.pt", trainer.ema.module, config, config.epochs, None)

    write_json(run_dir / "summary.json", dict(
        run=config.name, stage=config.stage, config=config.to_dict(), data=dataset_identity(data_dir),
        subset_rule=SUBSET_RULE, train_pulses=len(train_refs), monitor_pulses=len(monitor_refs),
        train_class_counts=dict(zip(CLASSES, class_counts.tolist())), parameters=parameters,
        training_precision=precision, data_load_seconds=round(load_seconds, 1),
        training_seconds=round(training_seconds, 1), best_epoch=best_epoch,
        best_monitor_mean_balanced_accuracy=best_score,
        peak_gpu_memory_bytes=torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None,
        peak_host_memory_bytes=peak_host_memory_bytes(), finished_utc=utc_now()))
    del trainer, train_x, monitor_x
    if device.type == "cuda":
        torch.cuda.empty_cache()
    evaluate_run(run_dir, data_dir, device, workers)
    save_training_curves(run_dir)

## Official-noise validation

In [ ]:
@torch.no_grad()
def predict_validation(model: nn.Module, data_dir: Path, refs: Sequence[PulseRef], device: torch.device,
                       batch_size: int, workers: int) -> tuple[np.ndarray, list[dict]]:
    """Predictions (pulses, 13 SNRs) and per-SNR noise diagnostics on official-noise examples."""
    model.eval()
    dataset = OfficialNoiseDataset(data_dir, refs, REQUESTED_SNRS)
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=workers, pin_memory=device.type == "cuda")
    snr_count = len(REQUESTED_SNRS)
    predictions = np.empty(len(dataset), dtype=np.int64)
    unattainable = np.zeros(snr_count, dtype=np.int64)
    clipped = np.zeros(snr_count, dtype=np.int64)
    started, done, next_report = time.perf_counter(), 0, 0.1
    for values, index, flags, clip_counts in loader:
        index = index.numpy()
        iq = interleaved_to_iq(values.to(device, non_blocking=True))
        predictions[index] = model(iq).argmax(dim=1).cpu().numpy()
        np.add.at(unattainable, index % snr_count, flags.numpy().astype(np.int64))
        np.add.at(clipped, index % snr_count, clip_counts.numpy())
        done += len(index)
        if done >= next_report * len(dataset):
            print(f"  {done}/{len(dataset)} validation examples ({time.perf_counter() - started:.0f} s)", flush=True)
            next_report += 0.1
    values_per_snr = len(refs) * VALUES_PER_PULSE
    diagnostics = [dict(requested_snr_db=snr, unattainable_examples=int(count),
                        clipped_value_fraction=float(clip) / values_per_snr)
                   for snr, count, clip in zip(REQUESTED_SNRS, unattainable, clipped)]
    return predictions.reshape(len(refs), snr_count), diagnostics


def evaluate_run(run_dir: Path, data_dir: Path, device: torch.device, workers: int,
                 checkpoint: str = "best", batch_size: int = 256) -> dict:
    """Score a checkpoint on every validation pulse at all 13 requested SNRs (official noise)."""
    configure_backends(inference=True)
    config, model, state = load_checkpoint(run_dir / f"checkpoint_{checkpoint}.pt", device)
    refs = load_split(data_dir)["validation"]
    print(f"Official-noise validation of {config.name} ({checkpoint} checkpoint, epoch {state['epoch']}): "
          f"{len(refs)} pulses x {len(REQUESTED_SNRS)} requested SNRs", flush=True)
    predictions, diagnostics = predict_validation(model, data_dir, refs, device, batch_size, workers)
    report = build_report(refs, predictions, diagnostics)
    report.update(run=config.name, checkpoint=checkpoint, checkpoint_epoch=state["epoch"])
    suffix = "" if checkpoint == "best" else f"_{checkpoint}"
    write_json(run_dir / f"validation_report{suffix}.json", report)
    if checkpoint == "best":
        record_validation(run_dir, report)
    write_validation_predictions(run_dir / f"validation_predictions{suffix}.csv", refs, predictions)
    save_report_plots(report, run_dir, suffix)
    print_report(report)
    return report

## Day 0: data check

In [ ]:
def published_train_pulses(seed: int, source_file: str) -> set[int]:
    """create_split_manifest.py rule: the 80 lowest SHA256('seed:source_file:pulse') hashes train."""
    def rank(pulse: int) -> tuple[bytes, int]:
        return hashlib.sha256(f"{seed}:{source_file}:{pulse}".encode()).digest(), pulse

    return set(sorted(range(PULSES_PER_RECORDING), key=rank)[:TRAIN_PER_RECORDING])


def inspect_pulse(data_dir: Path, file: str, pulse: int) -> dict:
    """Load one pulse as in the README; estimate its source SNR with the preprocessing windows."""
    recording = np.load(data_dir / file, mmap_mode="r", allow_pickle=False)
    raw = recording[0, pulse, :]
    iq = raw[0::2].astype(np.float32) + 1j * raw[1::2].astype(np.float32)
    power = np.abs(iq.astype(np.complex128)) ** 2
    noise = float(power[NOISE_WINDOW].mean())
    signal = float(power[SIGNAL_WINDOW].mean()) - noise
    snr = round(10 * math.log10(signal / noise), 1) if signal > 0 and noise > 0 else None
    return dict(file=file, pulse_index=pulse, shape=list(iq.shape), dtype=str(iq.dtype), source_snr_db=snr)


def check_command(args: argparse.Namespace) -> None:
    """Layout, manifest checksum and structure, split reproduction, recordings, labels, one pulse per class."""
    data_dir = Path(args.data).resolve()
    missing = [name for name in ("split_manifest.csv", "split_manifest.json")
               if not (data_dir / name).is_file()]
    for name in ("compact_dataset.py", "make_dataset_hackathon.py"):
        try:
            find_bundled_script(data_dir, name)
        except FileNotFoundError:
            missing.append(name)
    if missing:
        raise SystemExit(f"Missing: {', '.join(missing)} (looked in {data_dir} and {HACKATHON_DIR}). "
                         "Put the split files beside the recordings in dataset/.")
    if (data_dir / "INCOMPLETE.txt").exists():
        raise SystemExit("INCOMPLETE.txt found: the compact dataset is incomplete")

    manifest_status = verify_manifest_checksum(data_dir)
    print(f"split_manifest.csv SHA256: {manifest_status}")
    published = read_json(data_dir / "split_manifest.json")
    assignments: dict[str, dict[int, str]] = defaultdict(dict)
    for row in read_manifest(data_dir):
        file, pulse, split = row["file"], int(row["pulse_index"]), row["split"]
        if pulse in assignments[file] or split not in ("train", "validation"):
            raise ValueError(f"Duplicate pulse or unknown split: {file} pulse {pulse} {split!r}")
        assignments[file][pulse] = split
    expected_per_recording = Counter(train=TRAIN_PER_RECORDING, validation=VALIDATION_PER_RECORDING)
    for file, pulses in assignments.items():
        if set(pulses) != set(range(PULSES_PER_RECORDING)) or Counter(pulses.values()) != expected_per_recording:
            raise ValueError(f"{file}: needs pulses 0-99 with exactly 80 train and 20 validation")
    totals = Counter(split for pulses in assignments.values() for split in pulses.values())
    counts = {"files": len(assignments), "train": totals["train"], "validation": totals["validation"]}
    declared = {key: published[key] for key in counts}
    if counts != declared:
        raise ValueError(f"Manifest rows {counts} disagree with split_manifest.json {declared}")
    print(f"Split: {counts['files']} recordings, {counts['train']} train and {counts['validation']} "
          "validation pulses, 80/20 in every recording")
    if counts != OFFICIAL_RELEASE:
        print(f"Warning: the official release has {OFFICIAL_RELEASE}")

    source_files = {}
    for number, file in enumerate(sorted(assignments), 1):
        path = data_dir / file
        meta = read_json(path.with_suffix(".json"))
        recording = np.load(path, mmap_mode="r", allow_pickle=False)
        layout_ok = recording.shape == RECORDING_SHAPE and recording.dtype == np.int16
        del recording
        if (not layout_ok or meta.get("format") != "radarml-recorded-v1"
                or meta.get("source_pulse_indices") != list(range(PULSES_PER_RECORDING))):
            raise ValueError(f"{file}: unexpected array layout or JSON sidecar")
        if args.verify_checksums:
            if sha256_file(path) != meta["output_sha256"]:
                raise ValueError(f"{file}: SHA256 does not match its JSON sidecar")
            if number % 66 == 0:
                print(f"  hashed {number}/{len(assignments)} recordings", flush=True)
        label_from_filename(file)
        source_files[file] = meta["source_file"]
    print("Recordings: present with JSON sidecars, int16 (1, 100, 33600)"
          + (", SHA256 verified" if args.verify_checksums else ""))

    for file, pulses in assignments.items():
        training = {pulse for pulse, split in pulses.items() if split == "train"}
        if training != published_train_pulses(published["seed"], source_files[file]):
            raise ValueError(f"{file}: assignments do not reproduce the published algorithm")
    print(f"Assignments reproduced from SHA256('{published['seed']}:source_file:pulse_index')")

    class_counts = {split: Counter() for split in ("train", "validation")}
    for file, pulses in assignments.items():
        label = label_from_filename(file)
        for split in pulses.values():
            class_counts[split][label] += 1
    absent = [name for name in CLASSES if not class_counts["train"][name] or not class_counts["validation"][name]]
    if absent:
        raise ValueError(f"Classes missing from a split: {absent}")
    print("Pulses per class (train/validation): "
          + ", ".join(f"{name} {class_counts['train'][name]}/{class_counts['validation'][name]}" for name in CLASSES))

    examples = {}
    for file in sorted(assignments):
        label = label_from_filename(file)
        if label not in examples:
            first_train = min(pulse for pulse, split in assignments[file].items() if split == "train")
            examples[label] = inspect_pulse(data_dir, file, first_train)
    for label, example in sorted(examples.items()):
        snr = "n/a" if example["source_snr_db"] is None else f"{example['source_snr_db']:.1f} dB"
        print(f"  {label:<6} {example['file']} pulse {example['pulse_index']}: "
              f"{example['dtype']} {tuple(example['shape'])}, source SNR estimate {snr}")

    runs_dir = resolve_runs_dir(args)
    runs_dir.mkdir(parents=True, exist_ok=True)
    write_json(runs_dir / "data_check.json", dict(
        checked_utc=utc_now(), data=dataset_identity(data_dir), manifest_checksum=manifest_status,
        counts=counts, recording_checksums_verified=bool(args.verify_checksums),
        recordings_per_class=dict(sorted(Counter(label_from_filename(f) for f in assignments).items())),
        pulses_per_class={split: dict(sorted(counter.items())) for split, counter in class_counts.items()},
        example_pulses=examples))
    print(f"Data check passed; report in {runs_dir / 'data_check.json'}")

## Days 1-4: training, evaluation and selection

In [ ]:
def training_configs(args: argparse.Namespace) -> list[Config]:
    if args.config:
        configs = [Config.from_dict(read_json(Path(args.config)))]
    else:
        names = [name for preset in args.preset for name in PRESET_GROUPS.get(preset, (preset,))]
        configs = [PRESETS[name] for name in names]
    overrides = {key: value for key, value in (
        ("name", args.name), ("epochs", args.epochs), ("batch_size", args.batch_size),
        ("train_pulses_per_recording", args.train_pulses), ("seed", args.seed)) if value is not None}
    if "name" in overrides and len(configs) > 1:
        raise SystemExit("--name applies to a single run")
    return [dataclasses.replace(config, **overrides) for config in configs]


def train_command(args: argparse.Namespace) -> None:
    configs = training_configs(args)
    data_dir = Path(args.data).resolve()
    runs_dir = resolve_runs_dir(args)
    runs_dir.mkdir(parents=True, exist_ok=True)
    device = resolve_device(args.device)
    for config in configs:
        summary_path = runs_dir / config.name / "summary.json"
        if summary_path.is_file() and "validation" in read_json(summary_path):
            print(f"{config.name}: already trained and evaluated, skipped", flush=True)
            continue
        train_run(config, data_dir, runs_dir, device, args.workers)


def evaluate_command(args: argparse.Namespace) -> None:
    evaluate_run(Path(args.run).resolve(), Path(args.data).resolve(), resolve_device(args.device),
                 args.workers, args.checkpoint, args.batch_size)


def ranking_key(summary: dict) -> tuple[float, float]:
    """Event ranking: mean balanced accuracy over 13 SNRs, then the mean at -20/-25/-30 dB."""
    validation = summary["validation"]
    return validation["mean_balanced_accuracy"], validation["low_snr_mean_balanced_accuracy"]


def select_command(args: argparse.Namespace) -> None:
    """Rank evaluated runs and scale the best configuration to the final training budget."""
    runs_dir = resolve_runs_dir(args)
    summaries = [read_json(path) for path in sorted(runs_dir.glob(f"{args.pattern}/summary.json"))]
    ranked = sorted((summary for summary in summaries if summary.get("validation")), key=ranking_key, reverse=True)
    if not ranked:
        raise SystemExit(f"No evaluated runs match {runs_dir / args.pattern}")
    print(f"{'run':<20} {'mean BA':>8} {'-20..-30':>9} {'pulses':>7} {'epochs':>7}")
    for summary in ranked:
        validation, config = summary["validation"], summary["config"]
        print(f"{config['name']:<20} {validation['mean_balanced_accuracy']:>8.4f} "
              f"{validation['low_snr_mean_balanced_accuracy']:>9.4f} "
              f"{config['train_pulses_per_recording']:>7} {config['epochs']:>7}")
    best = Config.from_dict(ranked[0]["config"])
    final = PRESETS["final"]
    selected = dataclasses.replace(
        best, name=final.name, stage=final.stage, train_pulses_per_recording=final.train_pulses_per_recording,
        epochs=final.epochs, monitor_pulses_per_recording=final.monitor_pulses_per_recording,
        monitor_every=final.monitor_every)
    write_json(runs_dir / "selected_config.json", selected.to_dict())
    write_json(runs_dir / "selection.json", dict(
        created_utc=utc_now(), selected_run=best.name,
        rule="highest validation mean balanced accuracy, ties broken by the mean at -20/-25/-30 dB",
        ranking=[dict(run=summary["config"]["name"], **summary["validation"]) for summary in ranked]))
    print(f"Selected {best.name}; final settings written to {runs_dir / 'selected_config.json'}")

## Day 3: runtime and memory

In [ ]:
@torch.no_grad()
def time_inference(model: nn.Module, pulses: np.ndarray, batch_size: int, device: torch.device,
                   min_seconds: float = 3.0) -> dict:
    """Wall time from int16 interleaved rows in host memory to predicted labels on the host."""
    model.eval()
    batch_size = min(batch_size, len(pulses))
    starts = range(0, len(pulses) - batch_size + 1, batch_size)

    def run(start: int) -> None:
        batch = torch.from_numpy(pulses[start:start + batch_size]).to(device)
        model(interleaved_to_iq(batch)).argmax(dim=1).cpu()

    for _ in range(2):  # warm-up: kernel selection and allocator
        run(0)
    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    iterations, started = 0, time.perf_counter()
    while iterations < 3 or time.perf_counter() - started < min_seconds:
        run(starts[iterations % len(starts)])
        iterations += 1
    elapsed = time.perf_counter() - started
    count = iterations * batch_size
    result = dict(device=device.type, batch_size=batch_size, pulses=count,
                  ms_per_pulse=1000 * elapsed / count, pulses_per_second=count / elapsed)
    if device.type == "cuda":
        result["peak_gpu_memory_bytes"] = torch.cuda.max_memory_allocated(device)
    return result


def state_dict_bytes(model: nn.Module) -> int:
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.tell()


def benchmark_command(args: argparse.Namespace) -> None:
    """Inference time per pulse (preprocessing included), peak memory and model size."""
    run_dir = Path(args.run).resolve()
    data_dir = Path(args.data).resolve()
    configure_backends(inference=True)
    config, model, _ = load_checkpoint(run_dir / "checkpoint_best.pt", torch.device("cpu"))
    refs = evenly_spaced(load_split(data_dir)["validation"], 1)
    dataset = OfficialNoiseDataset(data_dir, refs, [0])
    pulses = np.stack([dataset[index][0].numpy() for index in range(min(512, len(dataset)))])
    devices = ([torch.device("cuda")] if torch.cuda.is_available() else []) + [torch.device("cpu")]
    timings = []
    for device in devices:
        model.to(device)
        batch_sizes = (1, 256) if device.type == "cuda" else (1, 32)
        for batch_size in batch_sizes:
            timing = time_inference(model, pulses, batch_size, device)
            timings.append(timing)
            print(f"{timing['device']:>4} batch {timing['batch_size']:>3}: {timing['ms_per_pulse']:.3f} ms per pulse "
                  f"({timing['pulses_per_second']:.0f} pulses/s)", flush=True)
    model.to("cpu")
    fastest = min(timing["ms_per_pulse"] for timing in timings)
    result = dict(
        run=config.name, recorded_utc=utc_now(),
        scope="int16 interleaved I/Q in host memory to label: conversion, RMS normalisation, lag product, network",
        precision="float32, TF32 disabled, deterministic cuDNN (same as validation and predict)",
        timings=timings, estimated_blind_test_minutes=round(fastest * BLIND_TEST_EXAMPLES / 60_000, 1),
        parameters=sum(parameter.numel() for parameter in model.parameters()),
        model_state_bytes=state_dict_bytes(model), peak_host_memory_bytes=peak_host_memory_bytes(),
        environment=environment_info(devices[0]))
    write_json(run_dir / "benchmark.json", result)
    print(f"Model {result['parameters']:,} parameters, {format_bytes(result['model_state_bytes'])}; "
          f"blind test (171,600 pulses) about {result['estimated_blind_test_minutes']} min of compute; "
          f"peak host memory {format_bytes(result['peak_host_memory_bytes'])}")
    print(f"Written {run_dir / 'benchmark.json'}")

## Day 4: model freeze

In [ ]:
def frozen_requirements() -> str:
    lines = [f"# Python {platform.python_version()} on {platform.platform()}",
             f"numpy=={np.__version__}",
             f"torch=={torch.__version__}"]
    matplotlib_version = package_version("matplotlib")
    if matplotlib_version:
        lines.append(f"matplotlib=={matplotlib_version}  # plots only, not needed by predict")
    if torch.version.cuda:
        lines.append(f"# CUDA {torch.version.cuda}, cuDNN {torch.backends.cudnn.version()}; "
                     "CUDA builds of torch install from the PyTorch package index")
    return "\n".join(lines) + "\n"


def method_summary(config: Config, summary: dict, report: dict, benchmark: dict | None, environment: dict) -> str:
    """Factual method and reproducibility summary generated from the frozen run."""
    inputs = "per-pulse RMS-normalised I/Q"
    if config.features == "iq_lag":
        inputs += " plus the lag-1 conjugate product z[n]*conj(z[n-1]) (phase difference weighted by magnitude)"
    sampling = "uniform" if config.low_snr_weight == 1.0 else f"weight {config.low_snr_weight:g} on SNR <= -10 dB"
    gru = "a bidirectional GRU over the downsampled sequence, " if config.temporal == "gru" else ""
    phase = "; random carrier-phase rotation" if config.random_phase else ""
    widths = "/".join(str(width) for width in config.widths)
    snrs = ", ".join(f"{snr:+d}" for snr in config.train_snrs)
    gpu = environment.get("gpu", {}).get("name", "none")
    lines = [
        "# RadarML Lite Track A (time domain): frozen pipeline",
        "",
        "## Method",
        f"- Input: ordered I/Q, 16,800 complex samples at 120 MS/s; {inputs}. "
        "No FFT, STFT, wavelet or spectrogram representation is used.",
        f"- Network: residual 1D CNN (stem kernel {config.stem_kernel}, stride 2; {len(config.widths)} stride-2 "
        f"stages, widths {widths}, squeeze-excitation), {gru}"
        f"attentive statistics pooling and an MLP head; {summary['parameters']:,} parameters.",
        f"- Training data: train split only, {config.train_pulses_per_recording} pulses per recording "
        f"({summary['train_pulses']:,} source pulses; {SUBSET_RULE}). No validation pulse is trained on.",
        f"- Training SNRs: requested {snrs} dB, {sampling}. "
        "Noise is added on the fly with the organizers' degrade rule (0-20 us noise window, 21-119 us signal "
        f"window, unattainable targets unchanged, int16 clipping){phase}.",
        f"- Objective: logit-adjusted cross-entropy (tau {config.logit_adjustment:g}) for balanced accuracy, "
        f"label smoothing {config.label_smoothing:g}.",
        f"- Optimisation: AdamW (learning rate {config.learning_rate:g}, weight decay {config.weight_decay:g}), "
        f"batch {config.batch_size}, {config.epochs} epochs, cosine schedule with {config.warmup_epochs:g}-epoch "
        f"warm-up, gradient clipping {config.grad_clip:g}, weight EMA {config.ema_decay:g}, "
        f"{summary['training_precision']} training precision.",
        f"- Frozen weights: EMA weights of epoch {summary['best_epoch']}, the best monitored validation score.",
        f"- Seeds: training {config.seed}; validation noise {EXPAND_SEED} (compact_dataset.py expand convention).",
        "",
        "## Validation results of the frozen model",
        f"{report['samples']:,} examples: every validation pulse at all 13 requested SNRs, generated with the "
        "bundled make_dataset_hackathon.degrade and the expand seeds. Confusion matrices (rows true, columns "
        "predicted) are in validation_report.json and confusion_matrices.png.",
        "",
        f"Mean balanced accuracy {report['mean_balanced_accuracy']:.4f}; "
        f"mean at -20/-25/-30 dB {report['low_snr_mean_balanced_accuracy']:.4f}.",
        "",
        "| Requested SNR (dB) | Balanced accuracy | Accuracy |",
        "| ---: | ---: | ---: |",
        *(f"| {row['requested_snr_db']:+d} | {row['balanced_accuracy']:.4f} | {row['accuracy']:.4f} |"
          for row in report["per_snr"]),
        "",
        "## Compute",
        f"- Hardware: {environment['cpu']} ({environment['logical_cpus']} logical CPUs), "
        f"{format_bytes(environment['ram_bytes'])} RAM, GPU {gpu}.",
        f"- Training time {format_duration(summary['training_seconds'])} plus {summary['data_load_seconds']:.0f} s "
        f"data loading; peak GPU memory {format_bytes(summary['peak_gpu_memory_bytes'])}.",
    ]
    if benchmark:
        lines += [f"- Inference on {timing['device']}, batch {timing['batch_size']}: "
                  f"{timing['ms_per_pulse']:.3f} ms per pulse, preprocessing included."
                  for timing in benchmark["timings"]]
        lines.append(f"- Model size: {benchmark['parameters']:,} parameters, "
                     f"{format_bytes(benchmark['model_state_bytes'])} (model.pt).")
    else:
        lines.append("- Inference time and model size: not benchmarked before the freeze.")
    lines += [
        f"- Software: Python {environment['python']}, numpy {environment['numpy']}, torch {environment['torch']}, "
        f"CUDA {environment['cuda'] or 'none'}.",
        "",
        "## Reused methods, code and assistance",
        "- ResNet (He et al., 2016); squeeze-and-excitation (Hu et al., 2018); attentive statistics pooling "
        "(Okabe et al., 2018); logit adjustment (Menon et al., 2021); AdamW (Loshchilov and Hutter, 2019).",
        "- Noise rule ported from make_dataset_hackathon.py (RadarML Lite, GPL-3.0); validation noise uses its degrade().",
        "- AI assistance: pipeline code written with an AI coding assistant (Claude, Anthropic).",
        "",
        "## Limitations",
        "- Validation measures new pulses from known configurations, not unseen configurations, receivers or environments.",
        "- Results are reported against requested SNR, not achieved in-pulse SNR.",
        "- Changes after the model freeze: none.",
    ]
    return "\n".join(lines) + "\n"


def freeze_command(args: argparse.Namespace) -> None:
    """Weights, configuration, code, dependency versions, validation results, inference command, hashes."""
    run_dir = Path(args.run).resolve()
    output = Path(args.output).resolve() if args.output else run_dir.parent / f"freeze_track_a_{run_dir.name}"
    for name in ("checkpoint_best.pt", "validation_report.json", "summary.json", "environment.json"):
        if not (run_dir / name).is_file():
            raise SystemExit(f"{run_dir / name} is missing; finish training and validation first")
    if output.exists() or output.with_name(output.name + ".zip").exists():
        raise SystemExit(f"{output} already exists; frozen artifacts are never overwritten")
    benchmark = read_json(run_dir / "benchmark.json") if (run_dir / "benchmark.json").is_file() else None
    if benchmark is None:
        print("Warning: no benchmark.json; run `benchmark` first so inference time and model size are recorded")
    code = CODE_FILE
    if not code.is_file():
        raise SystemExit(f"{code} is missing; freeze copies the script into the package, so keep track_a.py there")
    if sha256_file(code) != read_json(run_dir / "environment.json")["code_sha256"]:
        print("Warning: track_a.py changed after this run was trained; confirm the change does not affect inference")

    output.mkdir(parents=True)
    checkpoint = torch.load(run_dir / "checkpoint_best.pt", map_location="cpu", weights_only=True)
    config = Config.from_dict(checkpoint["config"])
    torch.save(checkpoint["state_dict"], output / "model.pt")
    write_json(output / "config.json", config.to_dict())
    shutil.copy2(code, output / "track_a.py")
    for name in ("validation_report.json", "validation_predictions.csv", "summary.json", "training_log.csv",
                 "environment.json", "benchmark.json", "ba_vs_snr.png", "confusion_matrices.png",
                 "training_curves.png"):
        if (run_dir / name).is_file():
            shutil.copy2(run_dir / name, output / name)
    (output / "requirements-frozen.txt").write_text(frozen_requirements(), encoding="utf-8")
    (output / "inference_command.txt").write_text(INFERENCE_INSTRUCTIONS, encoding="utf-8")
    report = read_json(run_dir / "validation_report.json")
    (output / "method_summary.md").write_text(
        method_summary(config, read_json(run_dir / "summary.json"), report, benchmark,
                       read_json(run_dir / "environment.json")), encoding="utf-8")

    hashes = {path.name: sha256_file(path) for path in sorted(output.iterdir()) if path.is_file()}
    (output / "SHA256SUMS").write_text("".join(f"{digest}  {name}\n" for name, digest in hashes.items()),
                                       encoding="utf-8")
    write_json(output / "freeze_manifest.json", dict(
        track="A", run=config.name, frozen_utc=utc_now(), checkpoint_epoch=checkpoint["epoch"],
        validation=validation_summary(report), git_commit=git_commit(), inference_command=INFERENCE_COMMAND,
        sha256=hashes))
    archive = Path(shutil.make_archive(str(output), "zip", root_dir=output.parent, base_dir=output.name))
    archive_digest = sha256_file(archive)
    archive.with_name(archive.name + ".sha256").write_text(f"{archive_digest}  {archive.name}\n", encoding="utf-8")
    print(f"Frozen pipeline: {output}")
    for name in FROZEN_FILES:
        print(f"  {name:<12} {hashes[name]}")
    print(f"Archive: {archive}\n  sha256       {archive_digest}")
    print(f"Inference command (inside the package): {INFERENCE_COMMAND}")

## Day 5 command, frozen on Day 4: blind-test inference

In [ ]:
def verify_package(package: Path) -> dict[str, str]:
    """The model, configuration and the running code must match the frozen SHA256SUMS."""
    listed = {}
    for line in (package / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
        if line.strip():
            digest, name = line.split(maxsplit=1)
            listed[name] = digest
    verified = {}
    for name in FROZEN_FILES:
        digest = sha256_file(package / name)
        if listed.get(name) != digest:
            raise SystemExit(f"{name} does not match SHA256SUMS: not the frozen artifact")
        verified[name] = digest
    if sha256_file(CODE_FILE) != verified["track_a.py"]:
        raise SystemExit("Run the track_a.py inside the freeze package")
    return verified


def read_template_ids(path: Path) -> list[str]:
    with path.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if not reader.fieldnames or reader.fieldnames[0] != "sample_id":
            raise ValueError(f"{path}: the first column must be sample_id")
        ids = [row["sample_id"] for row in reader]
    if not ids or len(ids) != len(set(ids)) or not all(ids):
        raise ValueError(f"{path}: sample IDs must be present, unique and non-empty")
    return ids


def validate_submission(path: Path, expected_ids: Sequence[str]) -> None:
    """The checks of score_submission.py: exact header, unique IDs, known labels, full coverage."""
    with path.open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["sample_id", "predicted_label"]:
            raise ValueError("Required columns: sample_id,predicted_label")
        rows = list(reader)
    ids = [row["sample_id"] for row in rows]
    if len(ids) != len(set(ids)) or set(ids) != set(expected_ids):
        raise ValueError("Prediction IDs must match the template exactly once each")
    if any(row["predicted_label"] not in CLASSES for row in rows):
        raise ValueError("Unknown or empty label")


@torch.no_grad()
def predict_command(args: argparse.Namespace) -> None:
    """Frozen inference: verify hashes, read test shards with participant_loader, write predictions.csv."""
    package = Path(args.package).resolve()
    verified = verify_package(package)
    device = resolve_device(args.device)
    configure_backends(inference=True)
    model = TrackANet(Config.from_dict(read_json(package / "config.json")))
    model.load_state_dict(torch.load(package / "model.pt", map_location="cpu", weights_only=True))
    model.to(device).eval()

    blind_dir = Path(args.blind_dir).resolve()
    samples = import_from_path(blind_dir / "participant_loader.py").Samples(blind_dir / "test")
    template_path = Path(args.template).resolve() if args.template else blind_dir / "submission_template.csv"
    template_ids = read_template_ids(template_path)
    output = Path(args.output).resolve()
    if output.exists():
        raise SystemExit(f"{output} exists; choose another --output")
    if len(samples) == 0:
        raise SystemExit("The test index is empty")

    print(f"Predicting {len(samples)} test examples on {device}", flush=True)
    predicted: dict[str, str] = {}
    inputs, ids = [], []
    started = time.perf_counter()
    for position in range(len(samples)):
        iq, info = samples[position]
        inputs.append(np.stack((iq.real, iq.imag)))
        ids.append(info["sample_id"])
        if len(inputs) == args.batch_size or position == len(samples) - 1:
            labels = model(torch.from_numpy(np.stack(inputs)).to(device)).argmax(dim=1).tolist()
            for sample_id, label in zip(ids, labels):
                if sample_id in predicted:
                    raise ValueError(f"Duplicate sample_id in the test index: {sample_id}")
                predicted[sample_id] = CLASSES[label]
            inputs, ids = [], []
        if (position + 1) % 20_000 == 0:
            print(f"  {position + 1}/{len(samples)} ({time.perf_counter() - started:.0f} s)", flush=True)
    seconds = time.perf_counter() - started

    expected_ids, predicted_ids = set(template_ids), set(predicted)
    missing, extra = expected_ids - predicted_ids, predicted_ids - expected_ids
    if missing or extra:
        raise ValueError(f"Template and test index disagree: {len(missing)} missing, {len(extra)} extra IDs")
    with output.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(["sample_id", "predicted_label"])
        writer.writerows((sample_id, predicted[sample_id]) for sample_id in template_ids)
    validate_submission(output, template_ids)
    write_json(output.with_name(output.stem + "_run.json"), dict(
        created_utc=utc_now(), package=str(package), verified_sha256=verified, blind_dir=str(blind_dir),
        template=str(template_path), device=str(device), examples=len(predicted), seconds=round(seconds, 1),
        ms_per_pulse=round(1000 * seconds / len(predicted), 4),
        predicted_label_counts=dict(sorted(Counter(predicted.values()).items()))))
    print(f"Wrote {output} ({len(predicted)} rows, {1000 * seconds / len(predicted):.3f} ms per pulse "
          "including loading and preprocessing)")

## Provenance

In [ ]:
def package_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def cpu_name() -> str:
    if sys.platform.startswith("linux"):
        try:
            for line in Path("/proc/cpuinfo").read_text().splitlines():
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
        except OSError:
            pass
    return platform.processor() or platform.machine()


def total_ram_bytes() -> int | None:
    try:
        import psutil
        return int(psutil.virtual_memory().total)
    except ImportError:
        pass
    try:
        return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
    except (AttributeError, OSError, ValueError):
        return None


def peak_host_memory_bytes() -> int | None:
    """Peak resident memory of this process (worker processes excluded)."""
    try:
        import resource
    except ImportError:  # Windows
        try:
            import psutil
        except ImportError:
            return None
        return int(getattr(psutil.Process().memory_info(), "peak_wset", 0)) or None
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return int(peak if sys.platform == "darwin" else peak * 1024)


def git_commit() -> str | None:
    try:
        result = subprocess.run(["git", "rev-parse", "HEAD"], cwd=SCRIPT_DIR,
                                capture_output=True, text=True, timeout=10)
    except (OSError, subprocess.SubprocessError):
        return None
    return result.stdout.strip() if result.returncode == 0 else None


def environment_info(device: torch.device) -> dict:
    info = dict(
        recorded_utc=utc_now(), python=platform.python_version(), platform=platform.platform(),
        cpu=cpu_name(), logical_cpus=os.cpu_count(), ram_bytes=total_ram_bytes(),
        numpy=np.__version__, torch=torch.__version__, matplotlib=package_version("matplotlib"),
        cuda=torch.version.cuda,
        cudnn=torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
        device=str(device), torch_threads=torch.get_num_threads(),
        code_sha256=sha256_file(CODE_FILE) if CODE_FILE.is_file() else None, git_commit=git_commit())
    if device.type == "cuda":
        properties = torch.cuda.get_device_properties(device)
        info["gpu"] = dict(name=properties.name, memory_bytes=properties.total_memory,
                           compute_capability=f"{properties.major}.{properties.minor}")
    return info

## Command line

In [ ]:
def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="RadarML Lite Track A (time domain) pipeline")
    commands = parser.add_subparsers(dest="command", required=True)

    def data_options(command: argparse.ArgumentParser) -> None:
        command.add_argument("--data", default=str(DEFAULT_DATA_DIR),
                             help=f"recorded dataset folder (default: {DEFAULT_DATA_DIR})")
        command.add_argument("--runs", help=f"runs folder (default: {DEFAULT_RUNS_DIR})")

    def device_options(command: argparse.ArgumentParser) -> None:
        command.add_argument("--device", help="torch device (default: cuda if available, else cpu)")
        command.add_argument("--workers", type=int, default=0,  # 0: notebook classes cannot be pickled to spawned workers
                             help="processes generating official validation noise")

    check = commands.add_parser("check", help="Day 0: verify layout, split, recordings and labels")
    data_options(check)
    check.add_argument("--verify-checksums", action="store_true",
                       help="also hash every recording against its JSON sidecar (reads 4.4 GB)")

    train = commands.add_parser("train", help="train, then score the full validation split")
    data_options(train)
    device_options(train)
    source = train.add_mutually_exclusive_group(required=True)
    source.add_argument("--preset", nargs="+", choices=sorted([*PRESETS, *PRESET_GROUPS]))
    source.add_argument("--config", help="JSON settings, e.g. selected_config.json")
    train.add_argument("--name", help="run name (single run only)")
    train.add_argument("--epochs", type=int)
    train.add_argument("--batch-size", type=int)
    train.add_argument("--train-pulses", type=int, help="training pulses per recording, 1-80")
    train.add_argument("--seed", type=int)

    evaluate = commands.add_parser("evaluate", help="re-score a checkpoint on the full validation split")
    data_options(evaluate)
    device_options(evaluate)
    evaluate.add_argument("--run", required=True)
    evaluate.add_argument("--checkpoint", choices=("best", "last"), default="best")
    evaluate.add_argument("--batch-size", type=int, default=256)

    select = commands.add_parser("select", help="Day 2: rank runs and write selected_config.json")
    data_options(select)
    select.add_argument("--pattern", default="day2-*", help="run folders to rank (glob)")

    benchmark = commands.add_parser("benchmark", help="Day 3: inference time, memory and model size")
    data_options(benchmark)
    benchmark.add_argument("--run", required=True)

    freeze = commands.add_parser("freeze", help="Day 4: build the frozen pipeline package")
    freeze.add_argument("--run", required=True)
    freeze.add_argument("--output", help="package folder (default: <runs>/freeze_track_a_<run>)")

    predict = commands.add_parser("predict", help="frozen inference on the blind test")
    predict.add_argument("--package", default=".", help="freeze folder (default: .)")
    predict.add_argument("--blind-dir", required=True, help="extracted blind_test folder")
    predict.add_argument("--template", help="default: <blind-dir>/submission_template.csv")
    predict.add_argument("--output", default="predictions.csv")
    predict.add_argument("--device", help="torch device (default: cuda if available, else cpu)")
    predict.add_argument("--batch-size", type=int, default=256)
    return parser


COMMANDS = {
    "check": check_command,
    "train": train_command,
    "evaluate": evaluate_command,
    "select": select_command,
    "benchmark": benchmark_command,
    "freeze": freeze_command,
    "predict": predict_command,
}


def main(argv: Sequence[str] | None = None) -> None:
    args = build_parser().parse_args(argv)
    COMMANDS[args.command](args)

## Run

Each cell is one command. Change the argument lists as needed. Frozen runs are never overwritten, so `freeze` fails if its output folder already exists.

### Day 0: check data layout, split and labels

In [ ]:
main(["check"])

### Day 1

In [ ]:
main(["train", "--preset", "day1"])

### Day 2

In [ ]:
main(["train", "--preset", "day2"])
main(["select"])

### Day 3

In [ ]:
# main(["benchmark", "--run", str(DEFAULT_RUNS_DIR / "<best day2 run>")])
# main(["train", "--config", str(DEFAULT_RUNS_DIR / "selected_config.json")])

### Day 4

In [ ]:
# main(["benchmark", "--run", str(DEFAULT_RUNS_DIR / "final")])
# main(["freeze", "--run", str(DEFAULT_RUNS_DIR / "final")])

### Day 5

`predict` is meant to run from inside the frozen package with its own `track_a.py` (`python track_a.py predict --package . --blind-dir <blind_test folder>`), so use the script there.